[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/04_llm_finetuning/lora_qlora.ipynb)

# 03 . LoRA and QLoRA from Scratch

## Concept — LoRA & QLoRA (revision notes)

**What it is**
- **LoRA (Low-Rank Adaptation)** freezes a pretrained weight `W` (shape `d_out x d_in`) and learns an additive update `delta W = (alpha / r) * B @ A`, where `A` is `r x d_in`, `B` is `d_out x r` and the rank `r` is tiny (4-64).
- **QLoRA** keeps the frozen base weights in 4-bit (NF4, blockwise-quantized) and trains LoRA adapters in 16/32-bit on top, dequantizing each weight on the fly during the forward pass.

**Why it matters**
- Trainable parameters drop from `d_out * d_in` to `r * (d_out + d_in)` per adapted matrix (often under 1% of the model), so gradients and Adam states shrink in the same proportion. Adapters are small files that can be swapped per task or merged for free. QLoRA additionally cuts the frozen weights to about 0.5 byte/param, which is what lets a 65B model be finetuned on one 48 GB GPU.

**When to use**
- Memory-limited finetuning, many task-specific variants of one base model, or when you want to limit forgetting. Use full finetuning when you have the memory and need maximum capacity (large behaviour shifts, continued pretraining).

**Math & intuition**
- Forward: `y = x W^T + (alpha/r) * x A^T B^T`. `B` is initialised to **zero** (and `A` randomly) so the model starts exactly equal to the base model.
- The hypothesis is that the weight change needed for a downstream task has low intrinsic rank, which we check below with an SVD of a real full-finetuning delta.
- `alpha / r` is a fixed scaling: changing `r` while keeping `alpha` constant keeps the update magnitude roughly stable, and a larger `alpha` is like a larger learning rate on the adapter.
- After training, `W' = W + (alpha/r) B A` gives a plain Linear with **zero inference overhead**.
- NF4 uses a 16-level codebook placed at quantiles of a standard normal (weights are roughly Gaussian), with a per-block absmax scale (e.g. block = 64 values). *Double quantization* also quantizes those scales to 8-bit.

### 1. Why low rank? Parameter counting for one matrix

**What this cell does (plain language):** we count the parameters of a single 512 x 512 weight versus its rank-r update for several ranks. The ratio `r*(d_in+d_out) / (d_in*d_out)` is the whole reason LoRA is cheap; it also shows when a high rank stops being a bargain.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, math, time, copy, random
torch.manual_seed(0)
d_in = d_out = 512
full = d_in * d_out
print(f"full matrix: {full:,} params")
for r in (1, 4, 8, 16, 64, 256):
    lora = r * (d_in + d_out)
    print(f"r={r:>3}: {lora:>8,} params = {100*lora/full:6.2f}% of full")
print("break-even rank (LoRA == full):", full // (d_in + d_out))

full matrix: 262,144 params
r=  1:    1,024 params =   0.39% of full
r=  4:    4,096 params =   1.56% of full
r=  8:    8,192 params =   3.12% of full
r= 16:   16,384 params =   6.25% of full
r= 64:   65,536 params =  25.00% of full
r=256:  262,144 params = 100.00% of full
break-even rank (LoRA == full): 256


### 2. A LoRA linear layer from scratch (with merge / unmerge)

**What this cell does (plain language):** we write `LoRALinear`: it wraps a frozen `nn.Linear`, adds trainable `A` (random init) and `B` (zero init), scales the update by `alpha/r`, and can `merge()` the update into the base weight (and `unmerge()` it). We verify three facts: the output equals the base at init, the merged layer matches the unmerged one, and unmerging restores the original weights.

In [2]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base, self.r, self.alpha = base, r, alpha
        self.scale = alpha / r
        for p in self.base.parameters(): p.requires_grad_(False)     # freeze pretrained weight
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))     # zero => delta W = 0 at start
        self.drop = nn.Dropout(dropout)
        self.merged = False
    def forward(self, x):
        out = self.base(x)
        if not self.merged:
            out = out + self.scale * (self.drop(x) @ self.A.T @ self.B.T)   # (x A^T) B^T : two thin matmuls
        return out
    @torch.no_grad()
    def delta_w(self): return self.scale * self.B @ self.A
    @torch.no_grad()
    def merge(self):
        if not self.merged: self.base.weight += self.delta_w(); self.merged = True
    @torch.no_grad()
    def unmerge(self):
        if self.merged: self.base.weight -= self.delta_w(); self.merged = False

torch.manual_seed(0)
lin = nn.Linear(32, 24); w0 = lin.weight.clone()
layer = LoRALinear(lin, r=4, alpha=8)
x = torch.randn(5, 32)
print("trainable:", [n for n, p in layer.named_parameters() if p.requires_grad])
print("output equals base at init:", torch.allclose(layer(x), F.linear(x, w0, lin.bias)))
with torch.no_grad(): layer.B.normal_(0, 0.1)                       # pretend training happened
y_unmerged = layer(x).clone()
layer.merge();   y_merged = layer(x).clone()
print("merged output == unmerged output:", torch.allclose(y_unmerged, y_merged, atol=1e-5))
layer.unmerge(); print("unmerge restores base weight:", torch.allclose(lin.weight, w0, atol=1e-5))
print("rank of delta W:", torch.linalg.matrix_rank(layer.delta_w()).item(), "(<= r = 4)")

trainable: ['A', 'B']
output equals base at init: True
merged output == unmerged output: True
unmerge restores base weight: True
rank of delta W: 4 (<= r = 4)


### 3. Gradients: why B starts at zero but still learns

**What this cell does (plain language):** we take one backward pass on a fresh adapter and print the gradient norms of `A` and `B`. With `B = 0`, the gradient w.r.t. `A` is exactly zero on step one (it is multiplied by `B`), while `B` receives a real gradient. After one update `B` is non-zero and `A` starts to learn too, which is why zero-init `B` is safe.

In [3]:
torch.manual_seed(0)
layer = LoRALinear(nn.Linear(32, 24), r=4, alpha=8)
target = torch.randn(5, 24)
F.mse_loss(layer(x), target).backward()
print("grad norm A:", layer.A.grad.norm().item(), "| grad norm B:", layer.B.grad.norm().item())
opt = torch.optim.SGD([layer.A, layer.B], lr=0.1); opt.step(); opt.zero_grad()
F.mse_loss(layer(x), target).backward()
print("after one step -> grad norm A:", layer.A.grad.norm().item(), "(now non-zero)")
print("base weight grad:", layer.base.weight.grad, "(frozen, no grad)")

grad norm A: 0.0 | grad norm B: 0.7842029929161072


after one step -> grad norm A: 0.08722618222236633 (now non-zero)
base weight grad: None (frozen, no grad)


### 4. Tiny GPT, 'pretraining' and the adaptation task

**What this cell does (plain language):** we reuse the tiny GPT and the two synthetic text domains from `full_finetuning.ipynb` (A = animal sentences for pretraining, B = space sentences to adapt to, plus a third domain C = food sentences used later for adapter swapping). We pretrain a base model on domain A for a short time, then evaluate it on all domains.

In [4]:
import math, torch, torch.utils.checkpoint, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass, asdict

@dataclass
class GPTConfig:
    vocab_size: int = 64
    block_size: int = 32
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64

class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.proj = nn.Linear(c.n_embd, c.n_embd)
        self.h = c.n_head
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # causal mask built in
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c.n_embd), nn.LayerNorm(c.n_embd)
        self.attn = Attn(c)
        self.mlp = nn.Sequential(nn.Linear(c.n_embd, 4 * c.n_embd), nn.GELU(), nn.Linear(4 * c.n_embd, c.n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c
        self.tok = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos = nn.Embedding(c.block_size, c.n_embd)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.use_ckpt = False
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)       # small init => starting loss ~ ln(vocab)
            if getattr(m, 'bias', None) is not None: nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None, ignore_index=-100):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for b in self.blocks:
            if self.use_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(b, x, use_reentrant=False)
            else:
                x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=ignore_index)
        return logits, loss

random.seed(0)
DOM = {
 "A": dict(adj=["big", "small", "red", "quick"], noun=["cat", "dog", "bird", "fish"], verb=["sees", "likes", "chases"]),
 "B": dict(adj=["old", "bright", "cold", "vast"], noun=["robot", "planet", "comet", "probe"], verb=["scans", "orbits", "finds"]),
 "C": dict(adj=["hot", "sweet", "salty", "fresh"], noun=["soup", "bread", "cake", "pie"], verb=["feeds", "warms", "fills"]),
}
def make_text(g, n, rng):
    return "".join(f"the {rng.choice(g['adj'])} {rng.choice(g['noun'])} {rng.choice(g['verb'])} the {rng.choice(g['noun'])}.\n" for _ in range(n))
rng = random.Random(0)
texts = {k: (make_text(g, 500, rng), make_text(g, 80, rng)) for k, g in DOM.items()}
chars = sorted(set("".join(a + b for a, b in texts.values())))
stoi = {c: i for i, c in enumerate(chars)}
enc = lambda s: torch.tensor([stoi[c] for c in s])
train_d = {k: enc(a) for k, (a, b) in texts.items()}; val_d = {k: enc(b) for k, (a, b) in texts.items()}

BLOCK = 32
def get_batch(data, bs, gen):
    ix = torch.randint(0, len(data) - BLOCK - 1, (bs,), generator=gen)
    return torch.stack([data[i:i + BLOCK] for i in ix]), torch.stack([data[i + 1:i + BLOCK + 1] for i in ix])

@torch.no_grad()
def evaluate(model, data, n=15):
    was = model.training; model.eval(); g = torch.Generator().manual_seed(123)
    out = sum(model(*get_batch(data, 16, g))[1].item() for _ in range(n)) / n
    model.train(was); return out

def train(model, data, steps, lr, bs=16, seed=0):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0); g = torch.Generator().manual_seed(seed); model.train()
    for s in range(steps):
        cur = lr * min(1, (s + 1) / 10) * (0.5 * (1 + math.cos(math.pi * s / steps)))   # warm-up x cosine
        for pg in opt.param_groups: pg["lr"] = cur
        loss = model(*get_batch(data, bs, g))[1]
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step()
    return model

cfg = GPTConfig(vocab_size=len(chars), block_size=BLOCK, n_layer=2, n_head=4, n_embd=64)
torch.manual_seed(0); base = GPT(cfg)
train(base, train_d["A"], steps=120, lr=3e-3)
base_state = copy.deepcopy(base.state_dict())
n_base = sum(p.numel() for p in base.parameters())
print("base params:", n_base)
print("base val loss ->", {k: round(evaluate(base, v), 3) for k, v in val_d.items()})

base params: 103808
base val loss -> {'A': 0.467, 'B': 3.598, 'C': 2.949}


### 5. Injecting LoRA into the transformer (and counting trainable parameters)

**What this cell does (plain language):** we write `add_lora(model, r, alpha, targets)` that walks the module tree and replaces every `nn.Linear` whose name matches a target (attention `qkv` / `proj` and the MLP layers) with a `LoRALinear`, freezes everything else, and reports trainable vs total parameters. This is what `peft.get_peft_model` does.

In [5]:
def add_lora(model, r=8, alpha=16, targets=("qkv", "proj", "mlp"), dropout=0.0):
    for p in model.parameters(): p.requires_grad_(False)             # freeze the entire base model first
    for name, mod in list(model.named_modules()):
        for cname, child in list(mod.named_children()):
            full_name = f"{name}.{cname}" if name else cname
            if isinstance(child, nn.Linear) and any(t in full_name for t in targets):
                setattr(mod, cname, LoRALinear(child, r, alpha, dropout))
    return model

def count(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    return tr, tot

m = GPT(cfg); m.load_state_dict(base_state); add_lora(m, r=8, alpha=16)
tr, tot = count(m)
print(f"trainable {tr:,} / total {tot:,} = {100*tr/tot:.1f}%")
print("adapted layers:", [n for n, mod in m.named_modules() if isinstance(mod, LoRALinear)][:4], "...")
xb = torch.randint(0, len(chars), (2, 16))
m0 = GPT(cfg); m0.load_state_dict(base_state)
print("still identical to base at init:", torch.allclose(m.eval()(xb)[0], m0.eval()(xb)[0], atol=1e-6))

trainable 16,384 / total 120,192 = 13.6%
adapted layers: ['blocks.0.attn.qkv', 'blocks.0.attn.proj', 'blocks.0.mlp.0', 'blocks.0.mlp.2'] ...
still identical to base at init: True


### 6. LoRA vs full finetuning on the new domain

**What this cell does (plain language):** we adapt the pretrained base to domain B twice with the same data and step budget: (1) full finetuning of all parameters, (2) LoRA with r=8. LoRA uses a higher learning rate (adapter-only training typically wants 10x or more). We compare domain-B loss, domain-A loss (forgetting), and trainable parameters.

In [6]:
STEPS = 80
full = GPT(cfg); full.load_state_dict(base_state); train(full, train_d["B"], STEPS, lr=3e-4)
lora = GPT(cfg); lora.load_state_dict(base_state); add_lora(lora, r=8, alpha=16); train(lora, train_d["B"], STEPS, lr=3e-3)
print(f"{'method':<8} | trainable | val B  | val A (forgetting)")
b = {"A": evaluate(base, val_d["A"]), "B": evaluate(base, val_d["B"])}
print(f"{'base':<8} | {0:>9} | {b['B']:.3f} | {b['A']:.3f}")
for name, mm in (("full", full), ("lora r=8", lora)):
    print(f"{name:<8} | {count(mm)[0]:>9,} | {evaluate(mm, val_d['B']):.3f} | {evaluate(mm, val_d['A']):.3f}")

method   | trainable | val B  | val A (forgetting)
base     |         0 | 3.598 | 0.467
full     |   103,808 | 1.307 | 1.864


lora r=8 |    16,384 | 1.111 | 2.168


### 7. Rank sweep and the alpha / r scaling

**What this cell does (plain language):** we vary the rank (1, 2, 4, 8, 16) at fixed `alpha = 2r` (so the scale stays 2) and see how final loss and parameter count move, then vary `alpha` at fixed rank to show it behaves like an adapter learning-rate multiplier. On tiny, easy tasks even a very low rank is enough.

In [7]:
print("rank sweep (alpha = 2r, lr 3e-3):")
for r in (1, 2, 4, 8, 16):
    mm = GPT(cfg); mm.load_state_dict(base_state); add_lora(mm, r=r, alpha=2 * r); train(mm, train_d["B"], STEPS, lr=3e-3)
    print(f"  r={r:>2} trainable={count(mm)[0]:>6,} val B={evaluate(mm, val_d['B']):.3f}")
print("alpha sweep (r = 4, lr 3e-3):")
for alpha in (1, 4, 16, 64):
    mm = GPT(cfg); mm.load_state_dict(base_state); add_lora(mm, r=4, alpha=alpha); train(mm, train_d["B"], STEPS, lr=3e-3)
    print(f"  alpha={alpha:>2} scale={alpha/4:>5.2f} val B={evaluate(mm, val_d['B']):.3f}")

rank sweep (alpha = 2r, lr 3e-3):


  r= 1 trainable= 2,048 val B=1.899


  r= 2 trainable= 4,096 val B=1.479


  r= 4 trainable= 8,192 val B=1.339


  r= 8 trainable=16,384 val B=1.110


  r=16 trainable=32,768 val B=0.926
alpha sweep (r = 4, lr 3e-3):


  alpha= 1 scale= 0.25 val B=1.989


  alpha= 4 scale= 1.00 val B=1.536


  alpha=16 scale= 4.00 val B=1.129


  alpha=64 scale=16.00 val B=0.982


### 8. Merging adapters, saving only the adapter, and swapping adapters

**What this cell does (plain language):** we (1) merge the trained LoRA weights into the base layers and check the logits are unchanged and the model is a plain GPT again, (2) save just the adapter tensors and compare file sizes, and (3) train a second adapter on domain C and swap between the two adapters on the same frozen base.

In [8]:
import io
def adapter_state(model): return {k: v for k, v in model.state_dict().items() if k.endswith(".A") or k.endswith(".B")}
def nbytes(sd):
    buf = io.BytesIO(); torch.save(sd, buf); return buf.tell()

lora.eval(); ref = lora(xb)[0].clone()
for mod in lora.modules():
    if isinstance(mod, LoRALinear): mod.merge()
print("merged logits == unmerged logits:", torch.allclose(lora(xb)[0], ref, atol=1e-5))
for mod in lora.modules():
    if isinstance(mod, LoRALinear): mod.unmerge()
print(f"adapter file {nbytes(adapter_state(lora))/1e3:.0f} KB vs full weights {nbytes(base.state_dict())/1e3:.0f} KB")

# swap adapters on one shared frozen base
adapters = {"B": adapter_state(lora)}
mC = GPT(cfg); mC.load_state_dict(base_state); add_lora(mC, r=8, alpha=16); train(mC, train_d["C"], STEPS, lr=3e-3)
adapters["C"] = adapter_state(mC)
shared = GPT(cfg); shared.load_state_dict(base_state); add_lora(shared, r=8, alpha=16)
for name, sd in adapters.items():
    shared.load_state_dict(sd, strict=False)                         # strict=False: adapter file has only A/B keys
    print(f"adapter {name} loaded -> val B {evaluate(shared, val_d['B']):.3f} | val C {evaluate(shared, val_d['C']):.3f}")

merged logits == unmerged logits: True
adapter file 71 KB vs full weights 424 KB


adapter B loaded -> val B 1.111 | val C 2.717
adapter C loaded -> val B 2.774 | val C 0.967


### 9. Is the full-finetuning update really low rank? SVD of delta W

**What this cell does (plain language):** we compute `delta W = W_finetuned - W_base` for a weight matrix from the fully finetuned model in section 6 and look at its singular values: how much of the update's energy (sum of squared singular values) is captured by the top few directions. This is the empirical motivation for LoRA, checked here on our own run rather than taken on faith.

In [9]:
wname = "blocks.0.mlp.0.weight"
dW = (dict(full.named_parameters())[wname] - base_state[wname]).detach()
S_ = torch.linalg.svdvals(dW)
energy = (S_ ** 2).cumsum(0) / (S_ ** 2).sum()
print("delta W shape:", tuple(dW.shape), "| max possible rank:", min(dW.shape))
for r in (1, 2, 4, 8, 16, 32):
    print(f"top-{r:>2} singular directions capture {100*energy[r-1].item():5.1f}% of the update energy")
# Best rank-r approximation error (Eckart-Young): relative Frobenius error equals the discarded energy
U, S_, Vh = torch.linalg.svd(dW, full_matrices=False)
r = 8; approx = (U[:, :r] * S_[:r]) @ Vh[:r]
print("relative error of the best rank-8 approximation:", ((dW - approx).norm() / dW.norm()).item())

delta W shape: (256, 64) | max possible rank: 64
top- 1 singular directions capture  19.9% of the update energy
top- 2 singular directions capture  38.3% of the update energy
top- 4 singular directions capture  64.0% of the update energy
top- 8 singular directions capture  84.5% of the update energy
top-16 singular directions capture  94.0% of the update energy
top-32 singular directions capture  97.8% of the update energy
relative error of the best rank-8 approximation: 0.3941364884376526


### 10. NF4-like blockwise 4-bit quantization from scratch

**What this cell does (plain language):** we build the 16-value NormalFloat codebook from normal-distribution quantiles, then quantize a Gaussian-ish weight matrix block by block: divide each block of 64 values by its absmax, snap every value to the nearest codebook entry (store a 4-bit index), and dequantize by `codebook[index] * absmax`. We compare reconstruction error with uniform int4 using the same blocks and report the memory per parameter, with and without double quantization of the scales.

In [10]:
from torch.distributions import Normal
def nf4_codebook():
    # bitsandbytes-style normal map: 8 positive + 7 negative quantiles + exact zero, scaled to [-1, 1]
    n = Normal(0.0, 1.0); off = 0.9677
    pos = n.icdf(torch.linspace(off, 0.5, 9)[:-1]); neg = -n.icdf(torch.linspace(off, 0.5, 8)[:-1])
    v = torch.cat([pos, torch.zeros(1), neg]).sort().values
    return v / v.abs().max()
NF4 = nf4_codebook(); INT4 = torch.linspace(-1, 1, 16)               # uniform levels for comparison
print("NF4 levels:", [round(x, 3) for x in NF4.tolist()])

def quantize_blockwise(w, code, block=64):
    flat = w.reshape(-1, block)                                       # each row = one block
    absmax = flat.abs().amax(dim=1, keepdim=True)                      # per-block scale
    idx = (flat / absmax).unsqueeze(-1).sub(code).abs().argmin(-1)     # nearest codebook entry (4-bit index)
    return idx.to(torch.uint8), absmax
def dequantize_blockwise(idx, absmax, code, shape):
    return (code[idx.long()] * absmax).reshape(shape)

torch.manual_seed(0)
W = torch.randn(256, 256) * 0.02                                       # Gaussian weights, like a trained layer
for name, code in (("NF4 ", NF4), ("int4", INT4)):
    idx, am = quantize_blockwise(W, code); Wq = dequantize_blockwise(idx, am, code, W.shape)
    print(f"{name}: mean abs error {(W - Wq).abs().mean():.5f} | relative Frobenius error {((W - Wq).norm()/W.norm()):.4f}")
bits_plain = 4 + 32 / 64                                              # 4-bit code + fp32 absmax per 64 values
bits_dq = 4 + 8 / 64 + 32 / (64 * 256)                                # absmax itself quantized to 8-bit (blocks of 256)
print(f"bits/param: fp16 16 | NF4 {bits_plain:.3f} | NF4 + double-quant {bits_dq:.3f} (about {16/bits_dq:.1f}x smaller than fp16)")

NF4 levels: [-1.0, -0.696, -0.525, -0.395, -0.284, -0.185, -0.091, 0.0, 0.08, 0.161, 0.246, 0.338, 0.441, 0.563, 0.723, 1.0]
NF4 : mean abs error 0.00145 | relative Frobenius error 0.0920
int4: mean abs error 0.00170 | relative Frobenius error 0.1004
bits/param: fp16 16 | NF4 4.500 | NF4 + double-quant 4.127 (about 3.9x smaller than fp16)


### 11. A QLoRA layer: frozen 4-bit base + trainable LoRA adapters

**What this cell does (plain language):** we write `QuantLinear`, which stores the base weight only as 4-bit indices + block scales (no float weight is kept), dequantizes on the fly in `forward`, and carries trainable LoRA matrices on top. We quantize the pretrained GPT's linear layers, train adapters on domain B, and compare against plain LoRA on the unquantized base. Indices are held one per byte here for clarity; real kernels pack two 4-bit values per byte.

In [11]:
class QuantLinear(nn.Module):
    def __init__(self, lin: nn.Linear, code=NF4, block=64, r=8, alpha=16):
        super().__init__()
        self.shape, self.code, self.scale = lin.weight.shape, code, alpha / r
        idx, am = quantize_blockwise(lin.weight.detach(), code, block)
        self.register_buffer("idx", idx); self.register_buffer("absmax", am)   # buffers: not trained, saved with the model
        self.register_buffer("bias", lin.bias.detach().clone())
        self.A = nn.Parameter(torch.randn(r, self.shape[1]) / math.sqrt(self.shape[1]))
        self.B = nn.Parameter(torch.zeros(self.shape[0], r))
    def forward(self, x):
        W = dequantize_blockwise(self.idx, self.absmax, self.code, self.shape)   # temporary full-precision copy
        return F.linear(x, W, self.bias) + self.scale * (x @ self.A.T @ self.B.T)

def add_qlora(model, r=8, alpha=16, targets=("qkv", "proj", "mlp")):
    for p in model.parameters(): p.requires_grad_(False)
    for name, mod in list(model.named_modules()):
        for cname, child in list(mod.named_children()):
            full_name = f"{name}.{cname}" if name else cname
            if isinstance(child, nn.Linear) and any(t in full_name for t in targets):
                setattr(mod, cname, QuantLinear(child, r=r, alpha=alpha))
    return model

q = GPT(cfg); q.load_state_dict(base_state); add_qlora(q)
print("val B right after quantizing (adapters still zero):", round(evaluate(q, val_d["B"]), 3), "| fp base:", round(b["B"], 3))
train(q, train_d["B"], STEPS, lr=3e-3)
print(f"QLoRA  val B {evaluate(q, val_d['B']):.3f} | LoRA (fp base) val B {evaluate(lora, val_d['B']):.3f}")
lin_bytes_fp32 = sum(p.numel() * 4 for n, p in base.named_parameters() if p.dim() == 2 and "blocks" in n)
lin_bytes_q = sum(b_.numel() * 0.5 for n, b_ in q.named_buffers() if n.endswith(".idx")) + sum(b_.numel() * 4 for n, b_ in q.named_buffers() if n.endswith("absmax"))
print(f"transformer-block linear weights: fp32 {lin_bytes_fp32/1e3:.0f} KB -> 4-bit codes + scales {lin_bytes_q/1e3:.0f} KB (packed estimate)")

val B right after quantizing (adapters still zero): 3.601 | fp base: 3.598


QLoRA  val B 1.109 | LoRA (fp base) val B 1.111
transformer-block linear weights: fp32 393 KB -> 4-bit codes + scales 55 KB (packed estimate)


## Common bugs

- **Forgetting to freeze the base weights.** If `requires_grad` stays True you are doing full finetuning and paying full memory. Fix: freeze first, then add adapters, and print trainable vs total parameters.
- **Initialising `B` randomly.** The model no longer equals the base at step 0, and the first updates fight a random perturbation. Fix: `B = 0`, `A` random (section 2 checks this).
- **Merging twice (or training after merging).** Merging adds `delta W` into `W` again, doubling the update, and a merged layer's adapter no longer affects the output. Fix: track a `merged` flag, `unmerge()` before more training.
- **Using the full-finetuning learning rate.** LoRA adapters typically need a much larger LR (1e-4 to 3e-3) because the update is scaled and low rank. A 1e-5 LR leaves the adapter near zero. Fix: sweep LR on a log scale.
- **Changing `r` but not `alpha`.** The effective scale `alpha/r` changes, so rank comparisons are confounded. Fix: keep `alpha/r` fixed (or use `alpha = 2r`) when sweeping rank, or use rsLoRA (`alpha/sqrt(r)`) at high ranks.
- **Merging into a quantized base.** You cannot add a float delta to 4-bit codes losslessly: merging means dequantize, add, requantize (adding rounding error) or keep the adapter separate. Fix: merge into a 16-bit copy of the base weights for deployment.
- **Not adapting enough modules.** Adapting only `q` and `v` is cheap but often worse than adapting all linear layers (QLoRA paper). Fix: target all attention and MLP projections if quality matters.
- **Loading an adapter onto a different base.** Adapter tensors only make sense for the exact base weights (and rank/targets) they were trained against. Fix: store the base model id/hash and `r`, `alpha`, `targets` next to the adapter.
- **Quantizing with one scale for the whole tensor.** A single outlier inflates the scale and crushes all other values to a few levels. Fix: blockwise scales (e.g. 64 values), which is why block size is a key QLoRA hyper-parameter.

## Exercises

**Exercise 1.** Compute the number of trainable parameters if LoRA (r=16) is applied only to the `qkv` and `proj` layers of every block of our GPT, and verify with `add_lora(..., targets=('qkv', 'proj'))`.

In [12]:
# Your attempt for Exercise 1 here

**Solution 1**

In [13]:
# Solution 1
d, L, r = cfg.n_embd, cfg.n_layer, 16
manual = L * (r * (d + 3 * d) + r * (d + d))        # qkv: d->3d, proj: d->d
m = GPT(cfg); m.load_state_dict(base_state); add_lora(m, r=16, alpha=32, targets=("qkv", "proj"))
print("manual:", manual, "| counted:", count(m)[0])

manual: 12288 | counted: 12288


**Exercise 2.** Add LoRA dropout (0.1) and check that outputs differ between two train-mode forward passes but are identical in eval mode.

In [14]:
# Your attempt for Exercise 2 here

**Solution 2**

In [15]:
# Solution 2
m = GPT(cfg); m.load_state_dict(base_state); add_lora(m, r=8, alpha=16, dropout=0.1)
for mod in m.modules():
    if isinstance(mod, LoRALinear): mod.B.data.normal_(0, 0.05)
m.train(); a1, a2 = m(xb)[0], m(xb)[0]
m.eval(); e1, e2 = m(xb)[0], m(xb)[0]
print("train-mode differ:", not torch.allclose(a1, a2), "| eval-mode identical:", torch.allclose(e1, e2))

train-mode differ: True | eval-mode identical: True


**Exercise 3.** Implement rsLoRA scaling (`alpha / sqrt(r)`) as an option and compare the effective scale for r = 4, 16, 64 with alpha = 16 against standard `alpha / r`.

In [16]:
# Your attempt for Exercise 3 here

**Solution 3**

In [17]:
# Solution 3
for r in (4, 16, 64):
    print(f"r={r:>2}: standard {16/r:.3f} | rsLoRA {16/math.sqrt(r):.3f}")
# rsLoRA keeps the update magnitude from shrinking as rank grows, so higher ranks actually help.

r= 4: standard 4.000 | rsLoRA 8.000
r=16: standard 1.000 | rsLoRA 4.000
r=64: standard 0.250 | rsLoRA 2.000


**Exercise 4.** Using the SVD of the delta W from section 9, build the best rank-4 approximation and measure how much val-B loss gets worse when you replace the full-finetuned weight by `W_base + rank-4 delta` for that one matrix.

In [18]:
# Your attempt for Exercise 4 here

**Solution 4**

In [19]:
# Solution 4
U, S_, Vh = torch.linalg.svd(dW, full_matrices=False)
approx = (U[:, :4] * S_[:4]) @ Vh[:4]
m = copy.deepcopy(full)
dict(m.named_parameters())[wname].data = (base_state[wname] + approx)
print("val B  full FT:", round(evaluate(full, val_d["B"]), 3), "| one matrix rank-4:", round(evaluate(m, val_d["B"]), 3))

val B  full FT: 1.307 | one matrix rank-4: 1.347


**Exercise 5.** Quantize with block sizes 16, 64, 256 and 4096 (NF4) and plot/print the reconstruction error. Which block size is best and what is the cost?

In [20]:
# Your attempt for Exercise 5 here

**Solution 5**

In [21]:
# Solution 5
for blk in (16, 64, 256, 4096):
    idx, am = quantize_blockwise(W, NF4, blk); Wq = dequantize_blockwise(idx, am, NF4, W.shape)
    print(f"block {blk:>4}: rel error {((W-Wq).norm()/W.norm()):.4f} | scale overhead {32/blk:.2f} bits/param")
# Smaller blocks track local magnitude better (lower error) but store more scales.

block   16: rel error 0.0810 | scale overhead 2.00 bits/param
block   64: rel error 0.0920 | scale overhead 0.50 bits/param
block  256: rel error 0.0989 | scale overhead 0.12 bits/param
block 4096: rel error 0.1111 | scale overhead 0.01 bits/param


**Exercise 6.** Add an outlier (value 1.0) to one block of W and compare the NF4 error of that block with 64-value blocks vs one scale for the whole tensor.

In [22]:
# Your attempt for Exercise 6 here

**Solution 6**

In [23]:
# Solution 6
W2 = W.clone(); W2[0, 0] = 1.0
for blk in (64, W2.numel()):
    idx, am = quantize_blockwise(W2, NF4, blk); Wq = dequantize_blockwise(idx, am, NF4, W2.shape)
    err = (W2 - Wq)[1:].abs().mean()          # error on all the non-outlier values
    print(f"block {blk:>6}: mean abs error on normal weights = {err:.5f}")

block     64: mean abs error on normal weights = 0.00145
block  65536: mean abs error on normal weights = 0.01545


**Exercise 7.** Merge the QLoRA adapters from section 11 into a dequantized copy of the base weights, build a plain GPT, and check its logits match the QLoRA model.

In [24]:
# Your attempt for Exercise 7 here

**Solution 7**

In [25]:
# Solution 7
plain = GPT(cfg); plain.load_state_dict(base_state)
sd = plain.state_dict()
for n, mod in q.named_modules():
    if isinstance(mod, QuantLinear):
        W_ = dequantize_blockwise(mod.idx, mod.absmax, mod.code, mod.shape) + mod.scale * mod.B @ mod.A
        sd[n + ".weight"] = W_.detach(); sd[n + ".bias"] = mod.bias
plain.load_state_dict(sd); plain.eval(); q.eval()
print("max logit diff:", (plain(xb)[0] - q(xb)[0]).abs().max().item())

max logit diff: 2.9802322387695312e-06
